# 04 Toolkit test: UK online retailer

**Goal:** prove the RFM toolkit works on a different shop, country, currency and data format, with one function call.
**Data:** UCI Online Retail II, a UK online gift retailer, about 1 million transactions, Dec 2009 to Dec 2011 (CC BY 4.0).

In [1]:
%pip install kagglehub

In [2]:
import kagglehub, shutil
from pathlib import Path

src = Path(kagglehub.dataset_download("mashlyn/online-retail-ii-uci"))
print(list(src.iterdir()))

[WindowsPath('~/.cache/kagglehub/datasets/mashlyn/online-retail-ii-uci/versions/3/online_retail_II.csv')]


In [3]:
dst = Path(r"..\data\raw\online_retail_ii")
dst.mkdir(parents=True, exist_ok=True)
shutil.copy(src / "online_retail_II.csv", dst / "online_retail_II.csv")

import pandas as pd
uk = pd.read_csv(dst / "online_retail_II.csv")
print(uk.shape)
uk.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
uk.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [5]:
uk.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

## Cleaning
One row = one product line in an invoice. Rules:
1. Drop rows with no Customer ID (cannot be segmented).
2. Drop cancellations (Invoice starting with "C", negative quantity).
3. Drop rows with price 0 or below.
4. Create `amount` = Quantity × Price, and convert InvoiceDate to a date.

In [6]:
rows_start = len(uk)

uk = uk[uk['Customer ID'].notna()]
after_customer = len(uk)

uk = uk[~uk['Invoice'].str.startswith('C') & (uk['Quantity'] > 0)]
after_cancel = len(uk)

uk = uk[uk['Price'] > 0].copy()
after_price = len(uk)

uk['InvoiceDate'] = pd.to_datetime(uk['InvoiceDate'])
uk['Customer ID'] = uk['Customer ID'].astype(int)
uk['amount'] = uk['Quantity'] * uk['Price']

print('Start:', rows_start)
print('No customer removed:', rows_start - after_customer)
print('Cancellations removed:', after_customer - after_cancel)
print('Price 0 removed:', after_cancel - after_price)
print('Clean rows:', len(uk), '| customers:', uk['Customer ID'].nunique(), '| invoices:', uk['Invoice'].nunique())

Start: 1067371
No customer removed: 243007
Cancellations removed: 18744
Price 0 removed: 71
Clean rows: 805549 | customers: 5878 | invoices: 36969


## Run the toolkit
First check how frequency is spread, then choose settings that fit this shop.

In [7]:
import sys
sys.path.append(r"..\src")
import rfm_toolkit as rt

rfm_uk = rt.build_rfm(uk, customer_col='Customer ID', date_col='InvoiceDate',
                      amount_col='amount', order_col='Invoice')
rfm_uk['frequency'].describe()

count    5878.000000
mean        6.289384
std        13.009406
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       398.000000
Name: frequency, dtype: float64

In [8]:
rt.PLAYBOOKS['uk'] = {
    'channels': {'Personal': 'Account manager email + phone', 'Semi-personal': 'Email', 'Automated': 'Email'},
    'retargeting': 'Google Ads and LinkedIn (business buyers)',
    'note': 'Mostly repeat business buyers: focus on reorder timing and volume deals.',
}

customers_uk, plan_uk = rt.run_rfm(
    uk, customer_col='Customer ID', date_col='InvoiceDate',
    amount_col='amount', order_col='Invoice',
    market='uk',
    freq_bins=(0, 1, 3, 10, float('inf')),
)
plan_uk[['customers', 'customer_share', 'revenue_share', 'median_spend', 'channel']]

Market: uk | Retargeting: Google Ads and LinkedIn (business buyers) | Mostly repeat business buyers: focus on reorder timing and volume deals.


,customers,customer_share,revenue_share,median_spend,channel
segment,,,,,
"At risk, high value",23,0.4,0.3,1708.2,Account manager email + phone
"New, high value",4,0.1,0.1,2046.8,Account manager email + phone
Champions,2113,35.9,73.5,2164.4,Account manager email + phone
"Loyal, slipping",2142,36.4,23.3,1017.3,Email
Hibernating,646,11.0,1.1,239.2,Email
New,234,4.0,0.4,237.8,Email
Lost,716,12.2,1.4,215.9,Email


In [9]:
scored_uk = rt.score_rfm(rfm_uk, freq_bins=(0, 1, 3, 10, float('inf')))
strict_uk = rt.assign_segments(scored_uk, repeat_min=3)
rt.build_action_plan(strict_uk, 'uk')[['customers', 'customer_share', 'revenue_share']]

Market: uk | Retargeting: Google Ads and LinkedIn (business buyers) | Mostly repeat business buyers: focus on reorder timing and volume deals.


,customers,customer_share,revenue_share
segment,,,
"At risk, high value",149,2.5,2.1
"New, high value",82,1.4,2.0
Champions,1590,27.1,70.0
"Loyal, slipping",1057,18.0,17.7
Hibernating,1273,21.7,3.2
New,679,11.6,2.0
Lost,1048,17.8,3.0


**Result:** the same toolkit segmented a UK wholesale-style retailer (5,878 customers, 36,969 invoices) with one call, after adapting two settings (frequency groups and a UK playbook).
The two shops are opposites. Olist: 97% one-time buyers; the value is in one-time high spenders (55% of revenue). UK: median 3 orders per customer; Champions are 35.9% of customers and 73.5% of revenue.
**Limitation found:** the segment rules were designed for a one-time-buyer business. In a repeat-buyer business the "new" and "at risk, high value" segments are almost empty, so the repeat threshold must be raised (repeat_min) or the rules extended. Testing on a second dataset exposed this, which is the point of testing.

**With stricter settings (repeat = 4+ orders):** Champions are 27.1% of customers and 70.0% of revenue; Loyal, slipping 18.0% and 17.7%.
The "high value" segments stay small (149 and 82 customers) because in this business high spenders are repeat buyers, so they sit in Champions and Loyal, not in the one-time segments. This is a real difference between the shops, not an error.

**Next improvement:** for repeat-buyer businesses, split Champions by value (e.g. VIP vs regular) so the highest spenders get account-manager treatment.